# 실습 1 신용카드 이상 탐지

데이터 준비
실습에 사용될 데이터는 Kaggle의 Credit Card Fraud Detection 데이터셋입니다. 이 데이터셋은 거래의 시간, 금액과 함께 28개의 PCA 변환된 특성들을 포함하고 있습니다. 'Class' 레이블은 사기 거래를 나타내는 1과 정상 거래를 나타내는 0으로 구분됩니다.

데이터를 불러오고, 전처리하는 기본적인 코드는 아래와 같습니다:

https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud?resource=download

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

# 데이터를 불러옵니다.
csv_data = pd.read_csv('creditcard.csv')
print(csv_data)

            Time         V1         V2        V3        V4        V5  \
0            0.0  -1.359807  -0.072781  2.536347  1.378155 -0.338321   
1            0.0   1.191857   0.266151  0.166480  0.448154  0.060018   
2            1.0  -1.358354  -1.340163  1.773209  0.379780 -0.503198   
3            1.0  -0.966272  -0.185226  1.792993 -0.863291 -0.010309   
4            2.0  -1.158233   0.877737  1.548718  0.403034 -0.407193   
...          ...        ...        ...       ...       ...       ...   
284802  172786.0 -11.881118  10.071785 -9.834783 -2.066656 -5.364473   
284803  172787.0  -0.732789  -0.055080  2.035030 -0.738589  0.868229   
284804  172788.0   1.919565  -0.301254 -3.249640 -0.557828  2.630515   
284805  172788.0  -0.240440   0.530483  0.702510  0.689799 -0.377961   
284806  172792.0  -0.533413  -0.189733  0.703337 -0.506271 -0.012546   

              V6        V7        V8        V9  ...       V21       V22  \
0       0.462388  0.239599  0.098698  0.363787  ... -0.01830

In [16]:
# target 데이터(class)의 분포를 확인합니다.
print("Class Distribution:")
print(csv_data['Class'].value_counts())

# 위 모든 feature를 사용해서, class를 예측
data = csv_data.loc[:, ~csv_data.columns.isin(['Time', 'Amount', 'Class'])]
target = csv_data.Class

Class Distribution:
Class
0    284315
1       492
Name: count, dtype: int64


In [30]:
# 데이터를 훈련 세트와 테스트 세트로 분할합니다.
from sklearn.model_selection import train_test_split
훈련용_data, 테스트용_data, 훈련용_target, 테스트용_target = train_test_split(
    data, target, test_size = 0.2, random_state = 40)

# 데이터 표준화 작업을 실시합니다,
from sklearn.preprocessing import StandardScaler
ss = StandardScaler()
ss.fit(훈련용_data)
ss.fit(테스트용_data)

표준화_훈련용_data = ss.transform(훈련용_data)
표준화_테스트용_data = ss.transform(테스트용_data)

In [31]:
# 로지스틱 회귀 모델을 생성하고 학습합니다.
from sklearn.linear_model import LogisticRegression
import numpy as np

lr = LogisticRegression()
lr.fit(표준화_훈련용_data, 훈련용_target)

# 학습된 모델로 테스트 데이터를 예측하고 평가합니다.
print(lr.predict(표준화_테스트용_data))


# 정확도, 정밀도, F1 Score를 계산 및 출력합니다.
from sklearn.metrics import classification_report

pred = lr.predict(표준화_테스트용_data)

print(classification_report(테스트용_target, pred))

# AUC 점수를 계산합니다.
lr_auc_score = roc_auc_score(테스트용_target, lr.predict_proba(표준화_테스트용_data)[:, 1])
print("Logistic Regression AUC score:", lr_auc_score)

[0 0 0 ... 0 0 0]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56865
           1       0.89      0.65      0.75        97

    accuracy                           1.00     56962
   macro avg       0.94      0.82      0.87     56962
weighted avg       1.00      1.00      1.00     56962

Logistic Regression AUC score: 0.9771161758587212


## Decision Tree 로 직접해보기

In [28]:
# 결정 트리 모델을 생성하고 학습합니다.
import numpy as np
from sklearn.model_selection import cross_validate , cross_val_score
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(random_state=10)

dt.fit(훈련용_data, 훈련용_target)

# 학습된 모델로 테스트 데이터를 예측하고 평가합니다.
print(dt.predict(테스트용_data))

# AUC 점수를 계산합니다.
dt_auc_score = roc_auc_score(테스트용_target, dt.predict_proba(테스트용_data)[:, 1])
print("Decision Tree AUC score:", dt_auc_score)

[0 0 0 ... 0 0 0]
Decision Tree AUC score: 0.9018947933294719


## Random Forest 로 해보기

In [29]:
# 랜덤 포레스트 모델을 생성하고 학습합니다.
from sklearn.ensemble import RandomForestClassifier
rfc = RandomForestClassifier(n_estimators = 10, n_jobs = -1, random_state = 40)

rfc.fit(훈련용_data, 훈련용_target)

# 학습된 모델로 테스트 데이터를 예측하고 평가합니다.
print(rfc.predict(테스트용_data))

# AUC 점수를 계산합니다.
rfc_auc_score = roc_auc_score(테스트용_target, rfc.predict_proba(테스트용_data)[:, 1])
print("Random Forest AUC score:", rfc_auc_score)

[0 0 0 ... 0 0 0]
Random Forest AUC score: 0.9482200110407991


## 퀴즈) SVM 사용해보기

In [32]:
# SVM 모델을 생성하고 학습합니다.
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.svm import SVC

# SVC 모델 생성 (기본 커널인 rbf를 사용하며, 확률 예측을 위해 probability=True 설정을 켭니다)
svm_model = SVC(kernel="rbf", probability=True, random_state=40)

# [중요] SVM은 거리를 기반으로 하므로 반드시 '표준화된' 데이터를 넣어야 합니다.
svm_model.fit(표준화_훈련용_data, 훈련용_target)

# 학습된 모델로 테스트 데이터를 예측하고 평가합니다.
pred_svm = svm_model.predict(표준화_테스트용_data)
print(pred_svm)

# 정확도, 정밀도, F1 Score 출력
print("\n--- SVM Classification Report ---")
print(classification_report(테스트용_target, pred_svm, digits=4))
print("---------------------------------------------")

# AUC 점수를 계산합니다.
svm_auc_score = roc_auc_score(
    테스트용_target, svm_model.predict_proba(표준화_테스트용_data)[:, 1]
)
print("SVM AUC score:", svm_auc_score)

[0 0 0 ... 0 0 0]

--- SVM Classification Report ---
              precision    recall  f1-score   support

           0     0.9996    1.0000    0.9998     56865
           1     0.9863    0.7423    0.8471        97

    accuracy                         0.9995     56962
   macro avg     0.9929    0.8711    0.9234     56962
weighted avg     0.9995    0.9995    0.9995     56962

---------------------------------------------
SVM AUC score: 0.9694514318139997


# 가장 AUC점수가 높았던 모델 GridSearch로 튜닝하기